In [1]:
import sys
print(sys.executable)

d:\banking77-triage\.venv\Scripts\python.exe


In [2]:
import datasets
print(datasets.__version__)

3.6.0


In [3]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset(
    "PolyAI/banking77",
    trust_remote_code=True
)

print(dataset)
print(dataset["train"][0])
print(dataset["train"].features)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 10003
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 3080
    })
})
{'text': 'I am still waiting on my card?', 'label': 11}
{'text': Value(dtype='string', id=None), 'label': ClassLabel(names=['activate_my_card', 'age_limit', 'apple_pay_or_google_pay', 'atm_support', 'automatic_top_up', 'balance_not_updated_after_bank_transfer', 'balance_not_updated_after_cheque_or_cash_deposit', 'beneficiary_not_allowed', 'cancel_transfer', 'card_about_to_expire', 'card_acceptance', 'card_arrival', 'card_delivery_estimate', 'card_linking', 'card_not_working', 'card_payment_fee_charged', 'card_payment_not_recognised', 'card_payment_wrong_exchange_rate', 'card_swallowed', 'cash_withdrawal_charge', 'cash_withdrawal_not_recognised', 'change_pin', 'compromised_card', 'contactless_not_working', 'country_support', 'declined_card_payment', 'declined_cash_withdrawal', 'declined_transfer',

In [4]:
train_df = dataset["train"].to_pandas()
test_df = dataset["test"].to_pandas()

label_feature = dataset["train"].features["label"]

train_df["intent"] = train_df["label"].apply(label_feature.int2str)
test_df["intent"] = test_df["label"].apply(label_feature.int2str)

train_df.head(10)

,text,label,intent
0,I am still waiting on my card?,11,card_arrival
1,What can I do if my card still hasn't arrived ...,11,card_arrival
2,I have been waiting over a week. Is the card s...,11,card_arrival
3,Can I track my card while it is in the process...,11,card_arrival
4,"How do I know if I will get my card, or if it ...",11,card_arrival
5,When did you send me my new card?,11,card_arrival
6,Do you have info about the card on delivery?,11,card_arrival
7,What do I do if I still have not received my n...,11,card_arrival
8,Does the package with my card have tracking?,11,card_arrival
9,I ordered my card but it still isn't here,11,card_arrival


In [5]:
print("Kích thước train:", train_df.shape)
print("Kích thước test:", test_df.shape)
print("Các cột train:", train_df.columns.tolist())
print("Số intent:", train_df["intent"].nunique())

train_df[["text", "intent"]].sample(15, random_state=42)

Kích thước train: (10003, 3)
Kích thước test: (3080, 3)
Các cột train: ['text', 'label', 'intent']
Số intent: 77


,text,intent
6883,Is it possible for me to change my PIN number?,change_pin
5836,I'm not sure why my card didn't work,declined_card_payment
8601,I don't think my top up worked,top_up_failed
2545,Can you explain why my payment was charged a fee?,card_payment_fee_charged
8697,How long does a transfer from a UK account tak...,balance_not_updated_after_bank_transfer
5573,Why am I getting declines when trying to make ...,declined_transfer
576,What is the $1 transaction on my account?,extra_charge_on_statement
6832,It looks like my card payment was sent back.,reverted_card_payment?
7111,Why am I unable to transfer money when I was a...,beneficiary_not_allowed
439,What if there is an error on the exchange rate?,card_payment_wrong_exchange_rate


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    train_df["text"],
    train_df["label"],
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

print("Train thực tế:", len(X_train))
print("Validation:", len(X_val))
print("Test giữ nguyên:", len(test_df))

Train thực tế: 8502
Validation: 1501
Test giữ nguyên: 3080


In [7]:
print("Số intent trong train:", y_train.nunique())
print("Số intent trong validation:", y_val.nunique())

print("\nTỉ lệ intent đầu tiên trong train:")
print(y_train.value_counts(normalize=True).head())

print("\nTỉ lệ intent đầu tiên trong validation:")
print(y_val.value_counts(normalize=True).head())

Số intent trong train: 77
Số intent trong validation: 77

Tỉ lệ intent đầu tiên trong train:
label
15    0.018701
28    0.018231
6     0.018113
75    0.017996
19    0.017643
Name: proportion, dtype: float64

Tỉ lệ intent đầu tiên trong validation:
label
15    0.018654
28    0.017988
75    0.017988
6     0.017988
19    0.017988
Name: proportion, dtype: float64


In [8]:
import time

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

baseline_nb = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("model", MultinomialNB())
])

start_time = time.perf_counter()

baseline_nb.fit(X_train, y_train)

train_time = time.perf_counter() - start_time

val_pred = baseline_nb.predict(X_val)

print(f"Training time: {train_time:.2f} seconds")
print(f"Validation accuracy: {accuracy_score(y_val, val_pred):.4f}")
print(f"Validation macro-F1: {f1_score(y_val, val_pred, average='macro'):.4f}")

Training time: 1.23 seconds
Validation accuracy: 0.7975
Validation macro-F1: 0.7631


In [9]:
from sklearn.metrics import classification_report

intent_names = label_feature.names

report = classification_report(
    y_val,
    val_pred,
    labels=range(len(intent_names)),
    target_names=intent_names,
    output_dict=True,
    zero_division=0
)

report_df = pd.DataFrame(report).T

print("Tổng quan:")
display(report_df.loc[["accuracy", "macro avg", "weighted avg"]])

print("10 intent có F1 thấp nhất:")
per_intent = report_df.loc[
    intent_names,
    ["precision", "recall", "f1-score", "support"]
].sort_values("f1-score")

display(per_intent.head(10))

Tổng quan:


,precision,recall,f1-score,support
accuracy,0.797468,0.797468,0.797468,0.797468
macro avg,0.820521,0.760610,0.763062,1501.000000
weighted avg,0.819338,0.797468,0.788061,1501.000000


10 intent có F1 thấp nhất:


,precision,recall,f1-score,support
contactless_not_working,0.000000,0.000000,0.000000,5.0
virtual_card_not_working,0.000000,0.000000,0.000000,6.0
card_swallowed,1.000000,0.111111,0.200000,9.0
card_acceptance,1.000000,0.111111,0.200000,9.0
lost_or_stolen_card,1.000000,0.250000,0.400000,12.0
card_not_working,0.636364,0.411765,0.500000,17.0
card_delivery_estimate,1.000000,0.352941,0.521739,17.0
exchange_rate,1.000000,0.411765,0.583333,17.0
transfer_timing,0.500000,0.736842,0.595745,19.0
compromised_card,1.000000,0.461538,0.631579,13.0


In [10]:
import time

from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

svm_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("model", LinearSVC(C=1.0, random_state=42))
])

start_time = time.perf_counter()

svm_model.fit(X_train, y_train)

svm_train_time = time.perf_counter() - start_time
svm_val_pred = svm_model.predict(X_val)

svm_accuracy = accuracy_score(y_val, svm_val_pred)
svm_macro_f1 = f1_score(y_val, svm_val_pred, average="macro")

print(f"Training time: {svm_train_time:.2f} seconds")
print(f"Validation accuracy: {svm_accuracy:.4f}")
print(f"Validation macro-F1: {svm_macro_f1:.4f}")

Training time: 3.77 seconds
Validation accuracy: 0.8767
Validation macro-F1: 0.8691


In [11]:
comparison_df = pd.DataFrame({
    "model": ["Naive Bayes", "Linear SVM"],
    "accuracy": [
        accuracy_score(y_val, val_pred),
        svm_accuracy
    ],
    "macro_f1": [
        f1_score(y_val, val_pred, average="macro"),
        svm_macro_f1
    ]
})

display(comparison_df.sort_values("macro_f1", ascending=False))

,model,accuracy,macro_f1
1,Linear SVM,0.876749,0.869111
0,Naive Bayes,0.797468,0.763062


In [12]:
from sklearn.metrics import classification_report

svm_report = classification_report(
    y_val,
    svm_val_pred,
    labels=range(len(intent_names)),
    target_names=intent_names,
    output_dict=True,
    zero_division=0
)

svm_report_df = pd.DataFrame(svm_report).T

print("Tổng quan Linear SVM:")
display(svm_report_df.loc[["accuracy", "macro avg", "weighted avg"]])

print("10 intent có F1 thấp nhất:")
svm_per_intent = svm_report_df.loc[
    intent_names,
    ["precision", "recall", "f1-score", "support"]
].sort_values("f1-score")

display(svm_per_intent.head(10))

Tổng quan Linear SVM:


,precision,recall,f1-score,support
accuracy,0.876749,0.876749,0.876749,0.876749
macro avg,0.879671,0.866603,0.869111,1501.000000
weighted avg,0.881467,0.876749,0.876046,1501.000000


10 intent có F1 thấp nhất:


,precision,recall,f1-score,support
virtual_card_not_working,0.750000,0.500000,0.600000,6.0
card_acceptance,0.714286,0.555556,0.625000,9.0
pin_blocked,0.833333,0.588235,0.689655,17.0
pending_transfer,0.708333,0.772727,0.739130,22.0
contactless_not_working,1.000000,0.600000,0.750000,5.0
transfer_not_received_by_recipient,0.791667,0.730769,0.760000,26.0
atm_support,0.769231,0.769231,0.769231,13.0
card_payment_fee_charged,0.807692,0.750000,0.777778,28.0
topping_up_by_card,0.800000,0.800000,0.800000,15.0
declined_cash_withdrawal,0.758621,0.846154,0.800000,26.0


In [13]:
svm_results = pd.DataFrame({
    "text": X_val,
    "true_intent": y_val.map(label_feature.int2str),
    "predicted_intent": pd.Series(
        svm_val_pred, index=y_val.index
    ).map(label_feature.int2str)
})

svm_errors = svm_results[
    svm_results["true_intent"] != svm_results["predicted_intent"]
]

print("Số ticket SVM đoán sai:", len(svm_errors))
display(svm_errors.sample(min(15, len(svm_errors)), random_state=42))

Số ticket SVM đoán sai: 185


,text,true_intent,predicted_intent
5587,I'm having issues with my card. You guys keep ...,declined_transfer,declined_cash_withdrawal
8839,I am seeing unathorized transactions in the ap...,cash_withdrawal_not_recognised,extra_charge_on_statement
7163,I didn't know there was a charge for tranferri...,transfer_fee_charged,card_payment_fee_charged
3107,WHAT IS THE ATMOSPHERE OF IT,card_acceptance,direct_debit_payment_not_recognised
36,How long does it take for a new card to ship?,card_arrival,card_delivery_estimate
840,"Hae there, I have a transaction that is in pro...",pending_cash_withdrawal,transaction_charged_twice
1764,I cannot use my PIN.,pin_blocked,get_physical_card
7990,How do I deposit cash?,top_up_by_cash_or_cheque,balance_not_updated_after_cheque_or_cash_deposit
7963,Who do I make a check out to if I want to top ...,top_up_by_cash_or_cheque,top_up_by_bank_transfer_charge
3195,Why won't my top up go through?,top_up_reverted,top_up_failed


In [14]:
print("svm_errors có tồn tại:", "svm_errors" in globals())

if "svm_errors" in globals():
    print("Các cột hiện có:", svm_errors.columns.tolist())

svm_errors có tồn tại: True
Các cột hiện có: ['text', 'true_intent', 'predicted_intent']


In [15]:
svm_results = pd.DataFrame({
    "text": X_val.to_numpy(),
    "true_intent": [
        label_feature.int2str(int(label))
        for label in y_val
    ],
    "predicted_intent": [
        label_feature.int2str(int(label))
        for label in svm_val_pred
    ]
})

svm_errors = svm_results[
    svm_results["true_intent"] != svm_results["predicted_intent"]
].copy()

top_confusions = (
    svm_errors
    .groupby(["true_intent", "predicted_intent"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Số ticket dự đoán sai:", len(svm_errors))
display(top_confusions.head(15))

Số ticket dự đoán sai: 185


,true_intent,predicted_intent,count
96,pin_blocked,get_physical_card,4
149,wrong_exchange_rate_for_cash_withdrawal,wrong_amount_of_cash_received,3
85,pending_cash_withdrawal,declined_cash_withdrawal,3
67,exchange_via_app,exchange_charge,3
36,card_payment_wrong_exchange_rate,wrong_exchange_rate_for_cash_withdrawal,3
121,transfer_fee_charged,card_payment_fee_charged,3
56,direct_debit_payment_not_recognised,card_payment_not_recognised,3
102,supported_cards_and_currencies,fiat_currency_support,3
26,card_not_working,declined_card_payment,2
64,exchange_rate,card_payment_wrong_exchange_rate,2


In [16]:
intent_support = svm_results["true_intent"].value_counts().rename("support")
intent_errors = svm_errors["true_intent"].value_counts().rename("error_count")

intent_error_analysis = pd.concat(
    [intent_support, intent_errors],
    axis=1
).fillna(0)

intent_error_analysis["error_rate"] = (
    intent_error_analysis["error_count"]
    / intent_error_analysis["support"]
)

display(
    intent_error_analysis
    .sort_values("error_rate", ascending=False)
    .head(15)
)

,support,error_count,error_rate
true_intent,,,
virtual_card_not_working,6,3.0,0.500000
card_acceptance,9,4.0,0.444444
pin_blocked,17,7.0,0.411765
contactless_not_working,5,2.0,0.400000
exchange_via_app,18,5.0,0.277778
transfer_not_received_by_recipient,26,7.0,0.269231
unable_to_verify_identity,15,4.0,0.266667
card_payment_fee_charged,28,7.0,0.250000
card_not_working,17,4.0,0.235294


In [17]:
print("svm_errors có tồn tại:", "svm_errors" in globals())

if "svm_errors" in globals():
    print("Các cột hiện có:", svm_errors.columns.tolist())

svm_errors có tồn tại: True
Các cột hiện có: ['text', 'true_intent', 'predicted_intent']


In [ ]:
import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# 1. Giữ đúng thứ tự của tập validation
texts = list(X_val)
y_true = np.asarray(y_val)

# 2. Dự đoán bằng pipeline đã huấn luyện
y_pred = svm_model.predict(texts)

# Nếu TF-IDF và SVM của bạn là hai đối tượng riêng,
# thay dòng predict phía trên bằng:
# X_val_tfidf = tfidf.transform(texts)
# y_pred = np.asarray(svm_model.predict(X_val_tfidf))
#
# Không dùng fit_transform() trên validation.

# 3. Ánh xạ nhãn số sang tên intent
label_names = list(label_names)
label_ids = np.arange(len(label_names))
id_to_name = dict(enumerate(label_names))

# Kiểm tra đầu vào
assert y_true.ndim == y_pred.ndim == 1
assert len(texts) == len(y_true) == len(y_pred) > 0
assert len(set(label_names)) == len(label_names)
assert np.isin(y_true, label_ids).all(), "Nhãn thật không khớp label_names"
assert np.isin(y_pred, label_ids).all(), "Nhãn dự đoán không khớp label_names"

print(f"Số mẫu validation: {len(y_true)}")
print(f"Accuracy: {accuracy_score(y_true, y_pred):.6f}")
print(
    "Macro-F1:",
    f"{f1_score(y_true, y_pred, labels=label_ids, average='macro', zero_division=0):.6f}"
)

# Thư mục lưu kết quả khi bạn chạy notebook
output_dir = Path("svm_error_analysis")
output_dir.mkdir(exist_ok=True)

NameError: name 'svm_pipeline' is not defined